In [1]:
import os, re, joblib, numpy as np, pandas as pd
from collections import Counter
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression, SGDClassifier
from sklearn.svm import LinearSVC
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import fbeta_score, f1_score
from sklearn.calibration import CalibratedClassifierCV
import warnings
warnings.filterwarnings('ignore')


def clean_text(text, keep_hyphens=False):
    """Базовая нормализация: нижний регистр, удаление лишних символов"""
    if not isinstance(text, str):
        return ""
    text = text.lower()
    pattern = r'[^a-zа-яё0-9\s\-]' if keep_hyphens else r'[^a-zа-яё0-9\s]'
    text = re.sub(pattern, ' ', text)
    return re.sub(r'\s+', ' ', text).strip()


def find_noise_words(corpus, df_thresh=0.70, min_len=2):
    """Определяет стоп-слова по частоте в корпусе + ручной список"""
    counts = Counter()
    n = len(corpus)
    for doc in corpus:
        counts.update(set(doc.split()))
    # частотные и слишком короткие
    noise = {w for w, c in counts.items() if (c / n) > df_thresh or len(w) < min_len}
    
    manual_noise = {
        'смотреть','онлайн','бесплатно','скачать','читать','слушать','играть','купить','заказать',
        'торрент','файл','трекер','стрим','трансляция','фильм','сериал','мультфильм','мультсериал',
        'аниме','дорама','книга','роман','рассказ','статья','видео','аудио','трек','песня','клип',
        'подкаст','сезон','серия','выпуск','эпизод','том','часть','hd','fullhd','720p','1080p','4k',
        'camrip','ts','webrip','brip','bluray','в','хорошем','качестве','озвучка','дубляж','субтитры',
        'перевод','оригинал','на','русском','английском','языке','голосовом','професиональном',
        'год','дата','час','часа','часов','мин','минут','сек','секунд','без','регистрации','смс',
        'и','а','но','или','же','ли','бы','то','что','как','где','когда','почему','зачем','кто',
        'для','от','до','через','над','под','о','об','у','к','по','с','со','из','за','в',
        'mkv','avi','mp4','mov','mp3','flac','wav','webm','srt','sub','dvdrip','brrip','webdl',
        'screener','scr','tc','ts','cam','hdtv','ppv','rip','full','uncut','director','cut',
        'extended','remastered','collection','box','set','pack','bundle','compilation',
        
        'этот', 'эта', 'это', 'эти', 'тех', 'те', 'та', 'тот', 'такой', 'такая', 'такое', 'такие',
        'все', 'всё', 'всего', 'вся', 'всех', 'всем', 'всеми', 'весь', 'всякий', 'всякая', 'всякое',
        'каждый', 'каждая', 'каждое', 'каждые', 'некоторый', 'некоторая', 'некоторое', 'некоторые',
        'другой', 'другая', 'другое', 'другие', 'иной', 'иная', 'иное', 'иные', 'сам', 'сама', 'само',
        'сами', 'весьма', 'очень', 'слишком', 'также', 'тоже', 'даже', 'только', 'лишь', 'почти',
        'уже', 'ещё', 'еще', 'уже', 'вдруг', 'наконец', 'вообще', 'например', 'может', 'можно',
        'нельзя', 'нужно', 'надо', 'будет', 'было', 'были', 'была', 'было', 'стал', 'стала', 'стали',
        'стало', 'является', 'являются', 'являлся', 'являлась', 'являлось', 'являлись',
        
        'сделать', 'создать', 'установить', 'настроить', 'загрузить', 'выгрузить', 'сохранить',
        'открыть', 'закрыть', 'показать', 'увидеть', 'посмотреть', 'прослушать', 'прочитать',
        'написать', 'нарисовать', 'спеть', 'танцевать', 'помогите', 'найти', 'найди', 'подскажите',
        'посоветуйте', 'рекомендуйте', 'хочу', 'хотел', 'хотела', 'нужен', 'нужна', 'ищу', 'ищет',
        'ищем', 'возможно', 'попробуйте', 'попробовать', 'получить', 'получите', 'подать', 'дайте',
        
        'ютуб', 'youtube', 'гугл', 'google', 'яндекс', 'yandex', 'википедия', 'wikipedia',
        'твиттер', 'twitter', 'фейсбук', 'facebook', 'инстаграм', 'instagram', 'тикток', 'tiktok',
        'телеграм', 'telegram', 'вк', 'vk', 'одноклассники', 'ok', 'mail', 'rambler', 'yahoo', 'bing',
        'пикабу', 'pikabu', 'реддит', 'reddit', 'discord', 'steam', 'epic games', 'gog', 'origin',
        'uplay', 'battle.net', 'netflix', 'hulu', 'disney+', 'amazon prime', 'ivi', 'megogo', 'okko',
        
        'игра', 'игры', 'геймплей', 'прохождение', 'летсплей', 'моды', 'мод', 'читы', 'чит', 'коды',
        'софт', 'программа', 'приложение', 'утилита', 'драйвер', 'обновление', 'патч', 'кряк', 'ключ',
        'серийник', 'лицензия', 'услуги', 'сервис', 'платформа', 'сайт', 'портал', 'блог', 'канал',
        'страница', 'группа', 'сообщество', 'паблик', 'чат', 'форум', 'тред', 'комментарий', 'пост',
        'запись', 'новость', 'новости', 'репортаж', 'интервью', 'дискуссия', 'обсуждение', 'опрос',
        'голосование', 'викторина', 'квиз', 'тест', 'задание', 'упражнение', 'задача', 'пример',
        'образец', 'шаблон', 'макет', 'дизайн', 'рисунок', 'картинка', 'фото', 'фотография',
        'изображение', 'графика', 'анимация', 'гифка', 'мем', 'демотиватор', 'комикс', 'манга',
        'ранобэ', 'фанфик', 'фанфикшн', 'ориджинал', 'кроссовер', 'сиквел', 'приквел', 'ремейк',
        'римейк', 'спин-офф', 'ответвление', 'адаптация', 'экранизация', 'постановка', 'спектакль',
        'мюзикл', 'опера', 'балет', 'концерт', 'фестиваль', 'выставка', 'экспозиция', 'музей',
        'галерея', 'библиотека', 'архив', 'хранилище', 'коллекция', 'собрание', 'каталог', 'база',
        'датасет', 'набор данных', 'выборка', 'корпус', 'текст', 'документ', 'папка', 'директория',
        'путь', 'ссылка', 'url', 'адрес', 'домен', 'хостинг', 'сервер', 'клиент', 'браузер',
        'расширение', 'плагин', 'аддон', 'модуль', 'библиотека', 'фреймворк', 'движок', 'ядро',
        'оболочка', 'интерфейс', 'gui', 'api', 'sdk', 'ide', 'редактор', 'компилятор', 'интерпретатор',
        'отладчик', 'профилировщик', 'тестер', 'анализатор', 'валидатор', 'конвертер', 'кодировщик',
        'декодировщик', 'шифратор', 'дешифратор', 'архиватор', 'распаковщик', 'загрузчик', 'выгрузчик',
        'инсталлятор', 'деинсталлятор', 'апдейтер', 'мигратор', 'синхронизатор', 'бэкапер', 'рестовер',
        'монитор', 'логировщик', 'логгер', 'бенчмарк', 'бенчмаркинг', 'тестирование', 'отладка',
        'профилирование', 'логирование', 'мониторинг', 'бэкап', 'восстановление', 'миграция',
        'синхронизация', 'архивация', 'распаковка', 'шифрование', 'дешифрование', 'кодирование',
        'декодирование', 'конвертация', 'транскодирование', 'рендеринг', 'компиляция', 'интерпретация',
        'линковка', 'сборка', 'деплой', 'релиз', 'паблишинг', 'дистрибуция', 'доставка', 'развертывание',
        'инсталляция', 'деинсталляция', 'апдейт', 'хотфикс', 'сервис-пак', 'beta', 'alpha', 'rc',
        'preview', 'nightly', 'dev', 'stable', 'версия', 'номер', 'ревизия', 'билд', 'сборка',
        'бинарник', 'исходник', 'код', 'скрипт', 'прога', 'программка', 'утилитка', 'тулза', 'инструмент',
        'средство', 'ресурс', 'актив', 'контент', 'материал', 'инфо', 'информация', 'данные', 'сведение',
        'факт', 'деталь', 'подробность', 'характеристика', 'параметр', 'свойство', 'атрибут', 'поле',
        'значение', 'величина', 'показатель', 'индекс', 'коэффициент', 'критерий', 'признак', 'симптом',
        'сигнал', 'помеха', 'артефакт', 'дефект', 'баг', 'глюк', 'ошибка', 'сбой', 'падение', 'краш',
        'фриз', 'лаг', 'тормоз', 'вылет', 'исключение', 'эксепшн', 'exception', 'error', 'warning',
        'notice', 'deprecated', 'fatal', 'critical', 'minor', 'major', 'blocker', 'trivial', 'enhancement',
        'feature', 'request', 'issue', 'task', 'subtask', 'epic', 'story', 'bugfix', 'hotfix', 'changeset',
        'commit', 'push', 'pull', 'merge', 'branch', 'tag', 'milestone', 'roadmap', 'backlog', 'sprint',
        'scrum', 'kanban', 'agile', 'waterfall', 'devops', 'ci', 'cd', 'pipeline', 'build', 'deploy',
        'test', 'qa', 'review', 'audit', 'check', 'verify', 'validate', 'certify', 'approve', 'reject',
        'rollback', 'fallback', 'failover', 'recover', 'backup', 'restore', 'archive', 'purge', 'clean',
        'wipe', 'format', 'partition', 'mount', 'unmount', 'attach', 'detach', 'link', 'unlink', 'symlink',
        'hardlink', 'copy', 'move', 'rename', 'delete', 'create', 'make', 'build', 'compile', 'assemble',
        'package', 'bundle', 'wrap', 'encapsulate', 'abstract', 'implement', 'override', 'overload',
        'extend', 'inherit', 'polymorph', 'encrypt', 'decrypt', 'encode', 'decode', 'serialize',
        'deserialize', 'marshal', 'unmarshal', 'parse', 'stringify', 'flatten', 'unflatten', 'zip',
        'unzip', 'tar', 'untar', 'gzip', 'gunzip', 'bzip2', 'bunzip2', 'xz', 'unxz', '7z', 'rar', 'unrar',
        'compress', 'decompress', 'minify', 'uglify', 'beautify', 'prettify', 'obfuscate', 'deobfuscate',
        'pack', 'unpack', 'box', 'unbox', 'convert', 'transcode', 'transform', 'map', 'reduce', 'filter',
        'collect', 'aggregate', 'group', 'sort', 'search', 'find', 'replace', 'regex', 'match', 'split',
        'join', 'slice', 'splice', 'concat', 'reverse', 'rotate', 'shuffle', 'sample', 'distinct', 'limit',
        'skip', 'offset', 'paginate', 'cursor', 'iterate', 'loop', 'foreach', 'while', 'for', 'break',
        'continue', 'return', 'exit', 'die', 'abort', 'halt', 'stop', 'pause', 'resume', 'start', 'launch',
        'init', 'boot', 'shutdown', 'reboot', 'restart', 'sleep', 'wait', 'notify', 'signal', 'interrupt',
        'poll', 'select', 'epoll', 'kqueue', 'iocp', 'async', 'sync', 'await', 'promise', 'future',
        'callback', 'listener', 'observer', 'emitter', 'dispatcher', 'handler', 'processor', 'worker',
        'thread', 'process', 'fiber', 'coroutine', 'task', 'job', 'batch', 'queue', 'stack', 'heap',
        'pool', 'cache', 'buffer', 'stream', 'channel', 'pipe', 'filter', 'interceptor', 'middleware',
        'decorator', 'adapter', 'facade', 'proxy', 'bridge', 'composite', 'strategy', 'template',
        'factory', 'builder', 'singleton', 'prototype', 'flyweight', 'command', 'chain', 'state',
        'visitor', 'mediator', 'memento', 'iterator', 'observer', 'subject', 'publisher', 'subscriber',
        'producer', 'consumer', 'supplier', 'runnable', 'callable', 'comparable', 'comparator', 'iterable',
        'collection', 'list', 'set', 'map', 'dictionary', 'multimap', 'bag', 'multiset', 'priority queue',
        'tree', 'graph', 'vertex', 'node', 'edge', 'path', 'cycle', 'connected component', 'spanning tree',
        'shortest path', 'longest path', 'maximum flow', 'minimum cut', 'bipartite matching', 'assignment',
        'transportation', 'transshipment', 'network flow', 'linear programming', 'integer programming',
        'mixed integer programming', 'constraint programming', 'satisfiability', 'optimization',
        'heuristic', 'metaheuristic', 'genetic algorithm', 'simulated annealing', 'tabu search',
        'ant colony', 'particle swarm', 'neural network', 'deep learning', 'machine learning',
        'data mining', 'pattern recognition', 'computer vision', 'natural language processing',
        'speech recognition', 'recommender system', 'information retrieval', 'search engine', 'database',
        'data warehouse', 'data lake', 'data mart', 'data fabric', 'data mesh', 'data vault',
        'data lineage', 'data catalog', 'data quality', 'data governance', 'data stewardship',
        'data integration', 'data transformation', 'data enrichment', 'data anonymization',
        'data pseudonymization', 'data masking', 'data encryption', 'data compression', 'data deduplication',
        'data versioning', 'data replication', 'data sharding', 'data partitioning', 'data indexing',
        'data clustering', 'data classification', 'data regression', 'data forecasting', 'data simulation',
        'data synthesis', 'data augmentation', 'data labeling', 'data annotation', 'data verification',
        'data validation', 'data cleaning', 'data wrangling', 'data munging', 'data profiling',
        'data auditing', 'data provenance', 'data dictionary', 'data glossary', 'data ontology',
        'data schema', 'data model', 'data architecture', 'data pipeline', 'data flow', 'data orchestration',
        'data scheduler', 'data monitor', 'data alert', 'data dashboard', 'data report',
        'data visualization', 'data storytelling', 'data literacy', 'data culture', 'data democratization',
        'data product', 'data as a service', 'data api', 'data contract', 'data sharing', 'data marketplace',
        'data exchange', 'data syndication', 'data feed', 'data stream', 'data event', 'data notification',
        'data trigger', 'data action', 'data workflow', 'data automation', 'data robotics',
        'data process automation', 'data intelligent', 'data insights', 'data analytics', 'data science',
        'data engineering', 'data ops', 'ml ops', 'ai ops', 'model ops', 'feature store', 'model registry',
        'model serving', 'model monitoring', 'model explainability', 'model fairness', 'model bias',
        'model drift', 'model decay', 'model retraining', 'model versioning', 'model lineage',
        'model governance', 'model audit', 'model validation', 'model certification', 'model deployment',
        'model rollback', 'model canary', 'model blue-green', 'model a/b testing', 'model multi-armed bandit'
    }
    return noise | manual_noise


class TextEnsemble:
    """Ансамбль из трёх различных TF-IDF + классификаторов"""
    def __init__(self, n_models=3):
        self.vecs = []
        self.clfs = []
        self.n_models = n_models

    def fit(self, X_texts, y):
        # три разные конфигурации векторизаторов
        vec_configs = [
            {'max_features': 50000, 'ngram_range': (1, 2), 'sublinear_tf': True, 'min_df': 2, 'max_df': 0.95, 'smooth_idf': True},
            {'max_features': 30000, 'ngram_range': (2, 4), 'sublinear_tf': True, 'min_df': 2, 'max_df': 0.95, 'smooth_idf': True},
            {'max_features': 40000, 'ngram_range': (3, 5), 'analyzer': 'char_wb', 'sublinear_tf': True, 'min_df': 2, 'max_df': 0.95, 'smooth_idf': True}
        ]
        # соответствующие классификаторы
        clf_configs = [
            LogisticRegression(C=1.5, max_iter=1500, class_weight='balanced', solver='lbfgs', n_jobs=-1),
            SGDClassifier(loss='log_loss', max_iter=1500, class_weight='balanced', random_state=42, n_jobs=-1),
            CalibratedClassifierCV(LinearSVC(C=1.0, class_weight='balanced', max_iter=3000, random_state=42), cv=3, method='isotonic')
        ]
        
        for i in range(min(self.n_models, len(vec_configs), len(clf_configs))):
            v = TfidfVectorizer(**vec_configs[i])
            X_vec = v.fit_transform(X_texts)
            clf = clf_configs[i]
            clf.fit(X_vec, y)
            self.vecs.append(v)
            self.clfs.append(clf)
        return self

    def predict_proba(self, X_texts):
        probs = np.zeros((len(X_texts), len(self.clfs[0].classes_)))
        for v, c in zip(self.vecs, self.clfs):
            probs += c.predict_proba(v.transform(X_texts))
        return probs / len(self.clfs)

    def predict(self, X_texts):
        return np.argmax(self.predict_proba(X_texts), axis=1)


def extract_title_improved(query, noise_words):
    """Удаляет шум и возвращает наиболее вероятный заголовок"""
    if not isinstance(query, str) or pd.isna(query):
        return np.nan
    q = query.strip().lower()
    
    quotes = re.findall(r'["«]([^"»]+)["»]', q)
    if quotes:
        return quotes[0].strip()
    
    # Удаление технических маркеров
    q = re.sub(r'\b\d{3,4}\b', ' ', q)                      # года
    q = re.sub(r'(?:s\d{2}e\d{2}|с\d{1,3}\s?е\d{1,3}|серия\s\d+|сезон\s\d+|эпизод\s\d+|выпуск\s\d+)', ' ', q)
    q = re.sub(r'\b(?:hd|fhd|720p|1080p|4k|camrip|bdrip|webdl|webrip|ts|tc|scr|dvdrip|avi|mkv|mp4|mov|mp3|flac)\b', ' ', q)
    
    # Очистка и токенизация
    q = re.sub(r'[^a-zа-яё0-9\s\-]', ' ', q)
    words = q.split()
    valid_words = [w for w in words if w not in noise_words and not w.isdigit() and len(w) > 1]
    if not valid_words:
        return np.nan
    
    # Группировка непрерывных валидных слов
    groups, cur = [], []
    for w in words:
        if w in noise_words or w.isdigit() or len(w) <= 1:
            if cur:
                groups.append(cur)
                cur = []
        else:
            cur.append(w)
    if cur:
        groups.append(cur)
    if not groups:
        return np.nan
    
    best = max(groups, key=len)
    title = " ".join(best).strip()
    return title if len(title) >= 2 else np.nan

def optimize_threshold(y_true, y_proba, beta=2):
    """Подбор порога для бинарной классификации по F_beta"""
    best_t, best_f2 = 0.5, -1
    for t in np.arange(0.10, 0.90, 0.01):
        pred = (y_proba[:, 1] >= t).astype(int)
        f2 = fbeta_score(y_true, pred, beta=beta)
        if f2 > best_f2:
            best_f2, best_t = f2, t
    return best_t

def calc_metrics(y_type_t, y_type_p, y_cont_t, y_cont_p, y_title_t, y_title_p, mask):
    """Вычисляет F2 для типа, макро F1 для контента и F1 для заголовка (по токенам)"""
    f2 = fbeta_score(y_type_t, y_type_p, beta=2)
    m1, t1 = 0.0, 0.0
    if mask.any():
        tc, pc = np.asarray(y_cont_t[mask]), np.asarray(y_cont_p[mask])
        valid_c = pd.notna(tc) & pd.notna(pc)
        if valid_c.any():
            labs = np.unique(np.concatenate([tc[valid_c].astype(str), pc[valid_c].astype(str)]))
            m1 = f1_score(tc[valid_c].astype(str), pc[valid_c].astype(str), average='macro', labels=labs, zero_division=0)
            
        tt, tp = np.asarray(y_title_t[mask]), np.asarray(y_title_p[mask])
        valid_t = pd.notna(tt) & pd.notna(tp)
        if valid_t.any():
            scores = []
            for a, b in zip(tt[valid_t], tp[valid_t]):
                sa, sb = set(str(a).lower().split()), set(str(b).lower().split())
                if not sa and not sb:
                    scores.append(1.0)
                elif not sa or not sb:
                    scores.append(0.0)
                else:
                    p = len(sa & sb) / len(sb)
                    r = len(sa & sb) / len(sa)
                    scores.append(2 * p * r / (p + r) if p + r > 0 else 0)
            t1 = np.mean(scores)
    
    return f2, m1, t1, 0.35 * f2 + 0.30 * m1 + 0.35 * t1



def run_pipeline():
    PATH = '/kaggle/input/datasets/antonoof/traindata-media/train.csv'
    DIR = '/kaggle/working/models'
    os.makedirs(DIR, exist_ok=True)

    df = pd.read_csv(PATH)
    df['QueryText_clean'] = df['QueryText'].apply(lambda x: clean_text(x, keep_hyphens=False))
    df['QueryText_title'] = df['QueryText'].apply(lambda x: clean_text(x, keep_hyphens=True))
    if 'Title' not in df.columns:
        df['Title'] = np.nan

    noise = find_noise_words(df['QueryText_clean'])
    joblib.dump(noise, os.path.join(DIR, 'noise_words.pkl'))
    print(f"   Найдено {len(noise)} стоп-слов.")
    
    skf = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)
    
    # Out-Of-Fold предсказания для оптимизации порога
    oof_type_true = np.zeros(len(df), dtype=int)
    oof_type_proba = np.zeros((len(df), 2))
    
    scores = {'f2': [], 'macro_f1': [], 'title_f1': [], 'combined': []}
    le_final, ens_cont_final = None, None
    df_pos = df[df['TypeQuery'] == 1]

    for fold, (tr_idx, va_idx) in enumerate(skf.split(df, df['TypeQuery'])):
        dtr, dva = df.iloc[tr_idx], df.iloc[va_idx]
        
        # TypeQuery классификатор
        ens_type = TextEnsemble(n_models=3).fit(dtr['QueryText_clean'], dtr['TypeQuery'])
        prob_type = ens_type.predict_proba(dva['QueryText_clean'])
        oof_type_proba[va_idx] = prob_type
        oof_type_true[va_idx] = dva['TypeQuery'].values

        # Классификатор ContentType и извлечение Title (только для TypeQuery=1)
        dtr_pos = dtr[dtr['TypeQuery'] == 1]
        le, ens_cont = None, None
        if not dtr_pos.empty:
            le = LabelEncoder()
            ens_cont = TextEnsemble(n_models=3).fit(dtr_pos['QueryText_clean'], le.fit_transform(dtr_pos['ContentType']))

        mask = dva['TypeQuery'] == 1
        y_cont_pred = pd.Series(np.nan, index=dva.index)
        y_title_pred = pd.Series(np.nan, index=dva.index)

        if ens_cont and mask.any():
            pos_idx = dva[mask].index
            pos_q_clean = dva.loc[pos_idx, 'QueryText_clean']
            pred_codes = ens_cont.predict(pos_q_clean)
            pred_ct = le.inverse_transform(pred_codes)
            y_cont_pred.loc[pos_idx] = pred_ct

            pos_q_title = dva.loc[pos_idx, 'QueryText_title']
            for idx, q in zip(pos_idx, pos_q_title):
                y_title_pred.loc[idx] = extract_title_improved(q, noise)

        # промежуточный порог для отчёта по фолду
        fold_t = optimize_threshold(dva['TypeQuery'], prob_type)
        y_type_pred = (prob_type[:, 1] >= fold_t).astype(int)

        f2, m1, t1, comb = calc_metrics(
            dva['TypeQuery'], y_type_pred,
            dva['ContentType'], y_cont_pred,
            dva['Title'], y_title_pred, mask
        )
        print(f"Fold {fold+1}: F2={f2:.4f} | MacroF1={m1:.4f} | TitleF1={t1:.4f} | Combined={comb:.4f} | FoldThresh={fold_t:.2f}")
        for k, v in zip(['f2','macro_f1','title_f1','combined'], [f2, m1, t1, comb]):
            scores[k].append(v)

    # Финальный порог на всех OOF-предсказаниях
    final_thresh = optimize_threshold(oof_type_true, oof_type_proba)
    print("\nСредние метрики CV:", {k: f"{np.mean(v):.4f}" for k, v in scores.items()})
    
    print("Финальное обучение на всех данных...")
    ens_type_final = TextEnsemble(n_models=3).fit(df['QueryText_clean'], df['TypeQuery'])
    
    if not df_pos.empty:
        le_final = LabelEncoder()
        ens_cont_final = TextEnsemble(n_models=3).fit(df_pos['QueryText_clean'], le_final.fit_transform(df_pos['ContentType']))

    # Сохранение моделей
    joblib.dump((ens_type_final, final_thresh), os.path.join(DIR, 'ens_type.pkl'))
    if ens_cont_final:
        joblib.dump(ens_cont_final, os.path.join(DIR, 'ens_content.pkl'))
        joblib.dump(le_final, os.path.join(DIR, 'le_content.pkl'))

if __name__ == "__main__":
    run_pipeline()

   Найдено 1011 стоп-слов.
Fold 1: F2=0.9673 | MacroF1=0.7765 | TitleF1=0.6936 | Combined=0.8143 | FoldThresh=0.29
Fold 2: F2=0.9666 | MacroF1=0.7686 | TitleF1=0.6897 | Combined=0.8103 | FoldThresh=0.31
Fold 3: F2=0.9694 | MacroF1=0.7546 | TitleF1=0.6903 | Combined=0.8073 | FoldThresh=0.28
Fold 4: F2=0.9678 | MacroF1=0.7524 | TitleF1=0.6816 | Combined=0.8030 | FoldThresh=0.28
Fold 5: F2=0.9689 | MacroF1=0.7824 | TitleF1=0.6795 | Combined=0.8117 | FoldThresh=0.28
Fold 6: F2=0.9707 | MacroF1=0.7352 | TitleF1=0.6508 | Combined=0.7881 | FoldThresh=0.32
Fold 7: F2=0.9692 | MacroF1=0.7133 | TitleF1=0.6670 | Combined=0.7867 | FoldThresh=0.27
Fold 8: F2=0.9641 | MacroF1=0.7684 | TitleF1=0.6882 | Combined=0.8089 | FoldThresh=0.26
Fold 9: F2=0.9689 | MacroF1=0.7495 | TitleF1=0.6710 | Combined=0.7988 | FoldThresh=0.26
Fold 10: F2=0.9693 | MacroF1=0.7374 | TitleF1=0.6789 | Combined=0.7981 | FoldThresh=0.30

Средние метрики CV: {'f2': '0.9682', 'macro_f1': '0.7538', 'title_f1': '0.6791', 'combined'

In [2]:
%%writefile solution.py
import os
import re
import joblib
import numpy as np
import pandas as pd
import gc

class PredictionModel:
    def __init__(self) -> None:
        self.dir = '/kaggle/working/models'
        self.ens_type, self.thresh = joblib.load(os.path.join(self.dir, 'ens_type.pkl'))
        self.ens_content = joblib.load(os.path.join(self.dir, 'ens_content.pkl'))
        self.le_content = joblib.load(os.path.join(self.dir, 'le_content.pkl'))
        self.noise = joblib.load(os.path.join(self.dir, 'noise_words.pkl'))

    def _clean_aggressive(self, text):
        if not isinstance(text, str): return ""
        text = text.lower()
        text = re.sub(r'[^a-zа-яё0-9\s]', ' ', text)
        return re.sub(r'\s+', ' ', text).strip()

    def _clean_title(self, text):
        if not isinstance(text, str): return ""
        text = text.lower()
        text = re.sub(r'[^a-zа-яё0-9\s\-]', ' ', text)
        return re.sub(r'\s+', ' ', text).strip()

    def _extract_title(self, query):
        if not isinstance(query, str) or pd.isna(query): return np.nan
        q = query.strip().lower()
        
        quotes = re.findall(r'["«]([^"»]+)["»]', q)
        if quotes: return quotes[0].strip()
        
        q = re.sub(r'\b\d{3,4}\b', ' ', q)
        q = re.sub(r'(?:s\d{2}e\d{2}|с\d{1,3}\s?е\d{1,3}|серия\s\d+|сезон\s\d+|эпизод\s\d+|выпуск\s\d+)', ' ', q)
        q = re.sub(r'\b(?:hd|fhd|720p|1080p|4k|camrip|bdrip|webdl|webrip|ts|tc|scr|dvdrip|avi|mkv|mp4|mov|mp3|flac)\b', ' ', q)
        q = re.sub(r'[^a-zа-яё0-9\s\-]', ' ', q)
        
        words = q.split()
        valid = [w for w in words if w not in self.noise and not w.isdigit() and len(w) > 1]
        if not valid: return np.nan
        
        groups, cur = [], []
        for w in words:
            if w in self.noise or w.isdigit() or len(w) <= 1:
                if cur: groups.append(cur); cur = []
            else: cur.append(w)
        if cur: groups.append(cur)
        if not groups: return np.nan
        
        best = max(groups, key=len)
        title = " ".join(best).strip()
        return title if len(title) >= 2 else np.nan

    def predict(self, df: pd.DataFrame) -> pd.DataFrame:
        out = df[["QueryText"]].copy()
        cleaned_agg = df['QueryText'].apply(self._clean_aggressive)
        
        probs = self.ens_type.predict_proba(cleaned_agg)
        out["TypeQuery"] = (probs[:, 1] >= self.thresh).astype(int)
        
        out["ContentType"] = np.nan
        out["Title"] = np.nan
        
        mask = out["TypeQuery"] == 1
        if mask.any():
            pos_idx = out[mask].index
            pos_q_agg = cleaned_agg.loc[pos_idx]
            pred_codes = self.ens_content.predict(pos_q_agg)
            pred_ct = self.le_content.inverse_transform(pred_codes)
            out.loc[pos_idx, "ContentType"] = pred_ct
            
            pos_q_title = df.loc[pos_idx, "QueryText"].apply(self._clean_title)
            for idx, q in zip(pos_idx, pos_q_title):
                out.loc[idx, "Title"] = self._extract_title(q)
                
        gc.collect()
        return out[["QueryText", "TypeQuery", "Title", "ContentType"]]

Writing solution.py


In [3]:
import numpy as np
import pandas as pd

from solution import PredictionModel

train_path = '/kaggle/input/datasets/antonoof/traindata-media/train.csv'

try:
    df_train = pd.read_csv(train_path)
except Exception as e:
    print(f"Ошибка загрузки: {e}")

model = PredictionModel()

predictions_df = model.predict(df_train)
predictions_df.head(5)

,QueryText,TypeQuery,Title,ContentType
0,белая королева сериал 2013,1,белая королева,сериал
1,омен 2006 смотреть онлайн на русском,1,омен,фильм
2,анонимный чат с фото,0,NaN,NaN
3,хищники против чужого скачать без торрента,1,хищники против чужого,фильм
4,скачать гравити фолз на 2 часа,1,гравити фолз,мультсериал


In [4]:
df_train.head(5)

,QueryText,TypeQuery,Title,ContentType
0,белая королева сериал 2013,1,белая королева,сериал
1,омен 2006 смотреть онлайн на русском,1,омен,фильм
2,анонимный чат с фото,0,NaN,NaN
3,хищники против чужого скачать без торрента,1,чужие против хищника,фильм
4,скачать гравити фолз на 2 часа,1,гравити фолз,мультсериал


In [5]:
import zipfile
import os

MODEL_DIR = '/kaggle/working/models'
zip_path = '/kaggle/working/models.zip'

with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zipf:
    for root, dirs, files in os.walk(MODEL_DIR):
        for file in files:
            file_path = os.path.join(root, file)
            arcname = os.path.relpath(file_path, start=os.path.dirname(MODEL_DIR))
            zipf.write(file_path, arcname)